<img align="left" src="https://scienceserver.linea.org.br/images/linea-logo.png" width=120 style="padding: 20px"> <br>
<img align="left" src="https://jupyter.org/assets/homepage/hublogo.svg" width=200 style="padding: 20px"> <br>
<br><br><br><br>

# Acesso ao Catálogo DES DR2 via TAP/pyvo

---

Bem vindo(a) ao LIneA JupyterHub!

O LIneA JupyterHub oferece acesso a dados públicos disponíveis online e a dados privados de levantamentos fotométricos cuja participação de cientistas brasileiros com _data rights_ é apoiada pelo LIneA. Neste notebook vamos exemplificar o acesso aos dados públicos do levantamento _Dark Energy Survey_ (DES) utilizando a biblioteca [`pyvo`](https://pyvo.readthedocs.io/) via protocolo TAP (_Table Access Protocol_).

Caso precise de ajuda, entre em contato: [helpdesk@linea.org.br](mailto:helpdesk@linea.org.br)

## 1. Sobre os dados

<img align="left" src="https://www.darkenergysurvey.org/wp-content/uploads/2016/01/des-logo-rev-lg.png" width=350 style="background-color:black; padding: 20px; margin-right: 2em">

O [Dark Energy Survey (DES)](https://www.darkenergysurvey.org/) é um levantamento fotométrico em 5 bandas do ótico ao infravermelho (_grizY_) que tem como principal objetivo a determinação da equação de estado da energia escura. O DES observou ~700 milhões de objetos em ~5000 graus quadrados no hemisfério sul durante 6 anos.

<img align="center" src="https://www.darkenergysurvey.org/wp-content/uploads/2021/06/dr2_footprint.png" width=500 style="padding: 20px"> <br>
_Figura: Footprint do DES Data Release 2_

O [segundo data release (DR2)](https://des.ncsa.illinois.edu/releases/dr2), com os dados dos seis anos de observação, está disponível publicamente e pode ser acessado pelo [LIneA User Query](https://userquery.linea.org.br/query) ou diretamente via TAP, como veremos a seguir.

## 2. Instalação das dependências

Execute a célula abaixo caso as bibliotecas não estejam instaladas no seu ambiente. No LIneA JupyterHub, `pyvo`, `requests` e `astropy` já estão disponíveis.

In [ ]:
# Descomente se precisar instalar
# !pip install pyvo requests astropy pandas matplotlib

## 3. Conexão com o banco de dados

### 3.1 Importações

In [ ]:
import os
import getpass
import time
from io import BytesIO

import pyvo
import requests
import pandas as pd
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table

### 3.2 Autenticação

| Ambiente | Token necessário? |
|---|---|
| LIneA JupyterHub | **Não** — pressione Enter em branco |
| Acesso externo (Jupyter local, Colab, etc.) | **Sim** — obtenha em [User Query → API Token](https://userquery.linea.org.br/query) |

> **Atenção:** nunca cole seu token diretamente no código. A célula abaixo usa  para ler o token de forma segura, sem exibi-lo na tela nem salvá-lo no notebook.

> **Limitação conhecida:** consultas **assíncronas** falham quando um token de autenticação está presente — é um bug no backend TAP do LIneA. Para uso em aula, recomenda-se deixar o token em branco e executar de dentro do JupyterHub LIneA, onde o acesso anônimo aos dados públicos do DES DR2 funciona sem restrições.

In [ ]:
# Tenta ler o token de uma variável de ambiente (opcional)
# token = os.environ.get("LINEA_TOKEN")

# if token is None:
#     resposta = getpass.getpass(
#         "Token de API LIneA (pressione Enter se estiver no JupyterHub): "
#     )
#     token = resposta.strip() or None  # None se Enter foi pressionado em branco

# if token:
#     print("Token informado. Autenticação habilitada.")
# else:
#     print("Sem token — conectando como usuário anônimo (JupyterHub LIneA).")

### 3.3 Criando a conexão TAP

In [ ]:
TAP_URL = "https://userquery.linea.org.br/tap"

session = requests.Session()
# if token:
#     session.headers["Authorization"] = f"Token {token}"

tap = pyvo.dal.TAPService(TAP_URL, session=session)
print(f"Conectado ao serviço TAP: {TAP_URL}")

## 4. Explorando o catálogo

### 4.1 Listando colunas da tabela `des_dr2.main`

O protocolo TAP disponibiliza metadados via `TAP_SCHEMA`. Podemos consultar quais colunas estão disponíveis com uma query SQL:

In [ ]:
query_schema = """
    SELECT column_name, datatype, description
    FROM TAP_SCHEMA.columns
    WHERE table_name = 'des_dr2.main'
    ORDER BY column_name
"""

colunas = tap.run_sync(query_schema).to_table().to_pandas()
print(f"{len(colunas)} colunas disponíveis")
colunas.head(20)

### 4.2 Primeira consulta

Vamos verificar a conexão recuperando os primeiros objetos da tabela:

In [ ]:
query_teste = """
    SELECT TOP 10
        coadd_object_id, ra, dec, mag_auto_g, mag_auto_r, mag_auto_i
    FROM des_dr2.main
"""

resultado = tap.run_sync(query_teste).to_table().to_pandas()
resultado

## 5. Tipos de consulta

### 5.1 Consultas síncronas

Recomendadas para consultas **rápidas** (retornam em segundos) ou com poucos resultados. O notebook aguarda a resposta antes de continuar.

```python
resultado = tap.run_sync(query)
df = resultado.to_table().to_pandas()
```

**Limitação:** o servidor TAP impõe um timeout (~30 segundos). Para queries maiores, use consultas assíncronas.

### 5.2 Consultas assíncronas

Para queries que retornam **muitos registros** ou que demoram mais de 30 segundos, use o modo assíncrono: o job é enviado ao servidor, executado em segundo plano, e o resultado é buscado ao final.

Filas disponíveis (`QUEUE`):
- `"default"` — até 30 segundos
- `"five_minutes"` — até 5 minutos
- `"two_hours"` — até 2 horas (para queries grandes)

O exemplo abaixo usa `TOP 100` para demonstrar o fluxo sem demorar. Para queries reais, remova o `TOP 100` e ajuste a fila conforme necessário.


> **Nota:** a consulta assíncrona requer acesso **sem token** (modo anônimo no JupyterHub LIneA). O uso de token de autenticação causa erro 404 no endpoint async — limitação conhecida do servidor TAP do LIneA.

In [ ]:
query_async = """
    SELECT TOP 100
        ra, dec, mag_auto_g, mag_auto_r, mag_auto_i
    FROM des_dr2.main
    WHERE mag_auto_g < 20
"""

# Submeter o job
job = tap.submit_job(query_async, QUEUE="five_minutes")
job.run()
print(f"Job enviado. ID: {job.job_id}")

# Aguardar a conclusão
while job.phase not in ("COMPLETED", "ERROR", "ABORTED"):
    print(f"Status: {job.phase}   ", end="\r")
    time.sleep(5)

print(f"Status final: {job.phase}")

# Buscar o resultado
if job.phase == "COMPLETED":
    result_url = f"{TAP_URL}/async/{job.job_id}/results/result"
    r = requests.get(result_url, headers=session.headers)
    df_async = Table.read(BytesIO(r.content), format="votable").to_pandas()
    print(f"{len(df_async):,} objetos retornados")
    display(df_async.head())
else:
    print(f"Erro na execução do job: {job.phase}")

## 6. Busca espacial com Q3C

O DES DR2 possui ~700 milhões de objetos. Uma busca por faixas de coordenadas (`WHERE ra BETWEEN x AND y`) faria uma varredura em toda a tabela, o que pode levar muitos minutos.

Para tirar proveito da indexação espacial do banco, usamos as funções da biblioteca [Q3C](https://github.com/segasai/q3c):

| Função | Uso |
|---|---|
| `CONTAINS(POINT(), CIRCLE())` | Cone search em torno de um ponto |
| `CONTAINS(POINT(), POLYGON())` | Região poligonal arbitrária |

### 6.1 Colunas utilizadas nas consultas

| Coluna | Descrição |
|---|---|
| `coadd_object_id` | Identificador único do objeto |
| `ra`, `dec` | Coordenadas equatoriais (graus) |
| `extended_class_coadd` | Classificação morfológica: 0–1 = estrelas, 2–3 = galáxias, -9 = sem dado |
| `flags_{g,r,i,z,y}` | Flags de qualidade — use `< 4` para fotometria confiável |
| `mag_auto_{g,r,i,z,y}_dered` | Magnitudes corrigidas pela extinção galáctica (recomendado) |
| `magerr_auto_{g,r,i}` | Erros nas magnitudes |

### 6.2 Alvo: Galáxia Anã de Sculptor

Vamos buscar estrelas na região da [Galáxia Anã de Sculptor](https://en.wikipedia.org/wiki/Sculptor_Dwarf_Galaxy), uma galáxia satélite da Via Láctea visível no hemisfério sul.

| Coordenada | Valor |
|---|---|
| Ascensão Reta | 01h 00m 09.3s |
| Declinação | −33° 42' 33" |

Usamos a classe `SkyCoord` do Astropy para converter as coordenadas para graus:

In [ ]:
sculptor = SkyCoord('01h00m09.3s', '-33d42m33s', frame='icrs')

ra0  = sculptor.ra.deg
dec0 = sculptor.dec.deg

print(f"RA:  {ra0:.4f} graus")
print(f"Dec: {dec0:.4f} graus")

### 6.3 Exemplo 1 — região retangular (`q3c_poly_query`)

Selecionamos uma região de 1° × 1° em torno de Sculptor (margem de 0.5° em cada direção).

Vértices (RA, Dec): (14.5, −34.2), (15.5, −34.2), (15.5, −33.2), (14.5, −33.2).

In [ ]:
query_1 = """
    SELECT coadd_object_id, ra, dec, flags_g,
           mag_auto_g_dered, mag_auto_r_dered, mag_auto_i_dered,
           magerr_auto_g, magerr_auto_r, magerr_auto_i
    FROM des_dr2.main
    WHERE CONTAINS(
              POINT('ICRS', ra, dec),
              POLYGON('ICRS', 14.5, -34.2, 15.5, -34.2, 15.5, -33.2, 14.5, -33.2)
          ) = 1
      AND extended_class_coadd < 2
"""

In [ ]:
%%time
dados_exemplo_1 = tap.run_sync(query_1).to_table().to_pandas()
print(f"{len(dados_exemplo_1):,} estrelas retornadas")
dados_exemplo_1.head()

### 6.4 Exemplo 2 — região circular (`q3c_radial_query`)

Selecionamos um cone de raio 0.5° em torno das coordenadas de Sculptor. Esta é a seleção que usaremos para o diagrama cor-magnitude.

In [ ]:
query_2 = f"""
    SELECT coadd_object_id, ra, dec, flags_g,
           mag_auto_g_dered, mag_auto_r_dered, mag_auto_i_dered,
           magerr_auto_g, magerr_auto_r, magerr_auto_i
    FROM des_dr2.main
    WHERE CONTAINS(
              POINT('ICRS', ra, dec),
              CIRCLE('ICRS', {ra0}, {dec0}, 0.5)
          ) = 1
      AND extended_class_coadd < 2
"""

In [ ]:
%%time
dados_exemplo_2 = tap.run_sync(query_2).to_table().to_pandas()
print(f"{len(dados_exemplo_2):,} estrelas retornadas")
dados_exemplo_2.head()

In [ ]:
# Exportar os dados para uso posterior
dados_exemplo_2.to_csv('dados_sculptor_DES.csv', index=False)
print("Arquivo salvo: dados_sculptor_DES.csv")

## 7. Visualizações

### 7.1 Distribuição espacial

Comparando a distribuição dos objetos nos dois exemplos:

In [ ]:
%%time
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)
fig.suptitle("Estrelas no DES DR2 — Região de Sculptor", fontsize=14)

for ax, dados_plot, titulo in zip(
    axes,
    [dados_exemplo_1, dados_exemplo_2],
    ["Exemplo 1: região retangular", "Exemplo 2: cone circular (r = 0.5°)"]
):
    ax.plot(dados_plot.ra, dados_plot.dec, 'k.', alpha=0.1, markersize=1)
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel("R.A. (graus)", fontsize=12)
    ax.set_ylabel("Dec. (graus)", fontsize=12)

plt.tight_layout()
plt.show()

### 7.2 Mapa de densidade

In [ ]:
%%time
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
fig.suptitle("Densidade de Estrelas — Região de Sculptor", fontsize=14)

for ax, dados_plot, titulo in zip(
    axes,
    [dados_exemplo_1, dados_exemplo_2],
    ["Exemplo 1: retangular", "Exemplo 2: circular"]
):
    h = ax.hist2d(dados_plot.ra, dados_plot.dec, bins=50, cmap='viridis')
    plt.colorbar(h[3], ax=ax, label="Densidade de pontos")
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel("R.A. (graus)", fontsize=12)
    ax.set_ylabel("Dec. (graus)", fontsize=12)

plt.tight_layout()
plt.show()

## 8. Limpeza e preparação dos dados

Para o CMD usaremos apenas o Exemplo 2 (cone circular). Primeiro liberamos da memória o dataframe que não precisamos mais:

In [ ]:
dados = dados_exemplo_2.copy()
del dados_exemplo_1
del dados_exemplo_2

Renomeamos as colunas para nomes mais curtos:

In [ ]:
dados.rename(columns={
    "coadd_object_id": "object_id",
    "mag_auto_g_dered": "mag_g",
    "mag_auto_r_dered": "mag_r",
    "mag_auto_i_dered": "mag_i",
    "magerr_auto_g":    "err_g",
    "magerr_auto_r":    "err_r",
    "magerr_auto_i":    "err_i",
}, inplace=True)

dados.info()

Calculamos a **cor g−r** (diferença de magnitudes entre duas bandas). Quanto menor o valor, mais azul a estrela; quanto maior, mais vermelha:

In [ ]:
dados["gmr"] = dados.mag_g - dados.mag_r
dados[["mag_g", "mag_r", "gmr"]].describe()

Filtramos a amostra para manter apenas estrelas com **fotometria confiável**:

- `flags_g < 4` — qualidade da extração fotométrica na banda _g_
- `mag != 99` — magnitudes com valor 99 indicam medição malsucedida

Aplicamos o mesmo critério para as bandas _r_ e _i_:

In [ ]:
dados.query(
    "flags_g < 4 & mag_g != 99. & mag_r != 99. & mag_i != 99.",
    inplace=True
)

print(f"{len(dados):,} estrelas após limpeza")
dados.describe()

## 9. Diagrama Cor-Magnitude (CMD)

O diagrama cor-magnitude é uma das ferramentas mais clássicas da astronomia. Cada estrela é plotada pela sua **cor** (eixo X, g−r) e seu **brilho** (eixo Y, magnitude _r_ — eixo invertido: menor = mais brilhante). Estrelas de mesma temperatura e idade ocupam posições semelhantes no diagrama, formando a chamada **sequência principal**.

Usamos `hexbin` com escala logarítmica de cor para visualizar a densidade de estrelas em cada região do diagrama:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6), dpi=150)

hb = ax.hexbin(
    dados.gmr, dados.mag_r,
    gridsize=500,
    bins='log',         # escala logarítmica de densidade
    cmap='viridis'
)

plt.colorbar(hb, ax=ax, label="Densidade de pontos")

ax.set_xlabel(r"$g - r$", fontsize=14)
ax.set_ylabel(r"$r$", fontsize=14)
ax.set_title("CMD — Estrelas na região de Sculptor (DES DR2)", fontsize=13)
ax.set_xlim(-1.5, 2.5)
ax.set_ylim(24, 16)    # eixo invertido: mais brilhante no topo

plt.tight_layout()
plt.savefig('cmd_sculptor_DES.png', dpi=150)
plt.show()
print(f"{len(dados):,} estrelas plotadas")

A sequência diagonal bem definida é a **sequência principal estelar** — estrelas de diferentes temperaturas (azuis à esquerda, vermelhas à direita) e luminosidades (brilhantes no topo, fracas na base). A estrutura compacta e nítida é característica de estrelas, que são objetos pontuais com fotometria precisa. Populações estelares distintas da Galáxia Anã de Sculptor podem aparecer como sobreposições à sequência principal da Via Láctea.

---

_Este notebook é baseado no repositório_ [jupyterhub-tutorial](https://github.com/linea-it/jupyterhub-tutorial)